In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from scipy.optimize import linprog
from time import time
import pandas as pd

In [3]:
# Statistics

Nlocations = 18159
current_AED_num = 8248

print("Number of locations:", Nlocations)
print("Twice the number of locations:", 2 * Nlocations)

Number of locations: 18159
Twice the number of locations: 36318


In [4]:
# Importing objective function

objective_fct = pd.read_csv('/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_matrices/objective_fct.csv')
objective_fct = objective_fct[["x"]].values
objective_fct = objective_fct.squeeze()

objective_fct.shape                             # want (Nlocations * 2,)

(36318,)

In [5]:
# Importing RHS

constraint_rhs = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_matrices/constraint_rhs.csv")
constraint_rhs = constraint_rhs[["x"]].values
constraint_rhs = constraint_rhs.squeeze()

constraint_rhs.shape    # want (Nlocations + 1,)

(18160,)

In [6]:
# resetting limit
pc_to_place = 100

total_limit = np.round(current_AED_num * pc_to_place / 100)
print("AEDs to place:", total_limit)

constraint_rhs[0] = total_limit

AEDs to place: 8248.0


In [ ]:
# Importing constraint matrix - TIME INTENSIVE

constraint_mat = pd.read_csv("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_matrices/constraint_mat_gradated.csv")
constraint_mat = constraint_mat.iloc[:,1:].values

constraint_mat.shape    # want (Nlocations + 1, Nlocations * 2 )

In [ ]:
# Performing linear programming

solution = linprog(c = objective_fct,
                   A_ub = constraint_mat[1:],
                   A_eq = constraint_mat[0:1],
                   b_ub = constraint_rhs[1:],
                   b_eq = constraint_rhs[0:1],
                   bounds = (0,3),
                   integrality= 1-objective_fct
                   )

In [ ]:
# Checking solution

print(solution.message)
print(max(solution.x)) # should be 3
print("Checking if placed correct number: " (sum(solution.x[:Nlocations]) == total_limit))

solution_name = "gradated_sol_" + str(total_limit) + "AEDs_" + str(Nlocations) + "locations"

np.savetxt("/content/drive/MyDrive/Backup Imperial Project Files/colab_files/LP_matrices/" + solution_name ".csv", solution.x)